# CLT-Forge Training on Custom NanoGPT

End-to-end notebook for training a Cross-Layer Transcoder (CLT) on your locally saved NanoGPT model.

**Your model hyperparameters:**
- `n_layer = 4`
- `n_head = 4`  
- `n_embd = 128`
- `block_size = 512`

**Pipeline stages:**
1. Install dependencies
2. Configuration — set your paths here
3. Load and inspect the NanoGPT checkpoint
4. Convert NanoGPT weights to TransformerLens HookedTransformer
5. Verify the converted model
6. Wrap model for CLT-Forge
7. Build CLT-Forge config
8. Precompute and cache MLP activations
9. Train the CLT
10. Evaluate reconstruction quality
11. Run AutoInterp (optional)
12. Compute Attribution Graph (optional)
13. Launch Visual Interface (optional)

---
## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Install CLT-Forge from your local clone (editable install)
CLT_FORGE_REPO_PATH = "/path/to/your/CLT-Forge"   # <-- CHANGE THIS to your local clone
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", CLT_FORGE_REPO_PATH, "-q"])

# Core dependencies
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformer_lens",   # HookedTransformer + NanoGPT weight conversion
    "tiktoken",           # GPT-2 tokeniser used by NanoGPT
    "datasets",           # HuggingFace datasets for activation caching
    "wandb",              # optional logging (disable via log_to_wandb=False)
    "zstandard",          # activation compression
])

print('All dependencies installed.')

---
## Cell 2 — Configuration

**Edit the paths in this cell before running anything else.**

In [ ]:
import os, torch

# ── Paths ──────────────────────────────────────────────────────────────────────
NANOGPT_CKPT_PATH      = "/path/to/your/nanogpt/ckpt.pt"   # <-- your saved .pt file
CONVERTED_MODEL_PATH   = "./converted_tl_model.pt"
CACHED_ACTIVATIONS_DIR = "./cached_activations"
CLT_CHECKPOINT_DIR     = "./clt_checkpoints"
AUTOINTERP_DIR         = "./autointerp_results"
ATTRIBUTION_GRAPH_DIR  = "./attribution_graphs"

for d in [CACHED_ACTIVATIONS_DIR, CLT_CHECKPOINT_DIR, AUTOINTERP_DIR, ATTRIBUTION_GRAPH_DIR]:
    os.makedirs(d, exist_ok=True)

# ── NanoGPT architecture (must match your training config exactly) ─────────────
N_LAYER    = 4
N_HEAD     = 4
N_EMBD     = 128
BLOCK_SIZE = 512      # context window
VOCAB_SIZE = 50304    # NanoGPT default (GPT-2 vocab padded to multiple of 64)
BIAS       = True     # True if you used bias=True in your model

# ── CLT training hyperparameters ───────────────────────────────────────────────
EXPANSION_FACTOR        = 16     # CLT features = N_EMBD * EXPANSION_FACTOR = 2048
CONTEXT_SIZE            = 64     # tokens per sample during caching (must be <= BLOCK_SIZE)
TOTAL_TRAINING_TOKENS   = 5_000_000
TRAIN_BATCH_SIZE_TOKENS = 2048
GRAD_ACCUM_STEPS        = 4
LEARNING_RATE           = 4e-4
L0_COEFFICIENT          = 2.0
USE_COMPRESSION         = True   # recommended: 4-7x storage reduction

# ── Dataset for activation caching ────────────────────────────────────────────
# Use any HuggingFace dataset tokenised with your tokeniser.
# This default works for NanoGPT trained with the standard GPT-2 tokeniser.
DATASET_PATH = "apollo-research/Skylion007-openwebtext-tokenizer-gpt2"

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device             : {DEVICE}")
print(f"NanoGPT layers     : {N_LAYER}")
print(f"NanoGPT n_embd     : {N_EMBD}")
print(f"CLT feature count  : {N_EMBD} x {EXPANSION_FACTOR} = {N_EMBD * EXPANSION_FACTOR}")

---
## Cell 3 — Load and Inspect the NanoGPT Checkpoint

In [ ]:
import torch

print(f"Loading checkpoint: {NANOGPT_CKPT_PATH}")
checkpoint = torch.load(NANOGPT_CKPT_PATH, map_location="cpu", weights_only=False)

# NanoGPT saves checkpoints as dicts with various metadata keys.
# The actual model weights live under checkpoint['model'].
if isinstance(checkpoint, dict):
    print("Top-level checkpoint keys:", list(checkpoint.keys()))
    # Print any stored training metadata
    for key in ("model_args", "iter_num", "best_val_loss", "config"):
        if key in checkpoint:
            print(f"  {key}: {checkpoint[key]}")
    # Extract the actual state dict
    if "model" in checkpoint:
        raw_sd = checkpoint["model"]
    elif "state_dict" in checkpoint:
        raw_sd = checkpoint["state_dict"]
    else:
        raw_sd = checkpoint  # the dict IS the state dict
else:
    # checkpoint is a nn.Module object
    raw_sd = checkpoint.state_dict()

# Strip 'module.' prefix added by DDP training
raw_sd = {(k[len("module."):] if k.startswith("module.") else k): v
          for k, v in raw_sd.items()}

print(f"\nParameter tensors: {len(raw_sd)}")
print("First 12 keys:")
for k in list(raw_sd.keys())[:12]:
    print(f"  {k:60s} {tuple(raw_sd[k].shape)}")

---
## Cell 4 — Convert NanoGPT Weights to TransformerLens Format

TransformerLens has a native NanoGPT weight conversion module. This cell performs the remapping manually
so it works with any NanoGPT checkpoint regardless of how it was saved.

Key shape transformations:
- NanoGPT `c_attn` packs Q, K, V into one `(3*d_model, d_model)` matrix — we split and reshape each to `(n_heads, d_model, d_head)`
- NanoGPT uses Conv1D convention `(out, in)` — TransformerLens uses `(in, out)` for W_in / W_out

In [ ]:
import torch
from transformer_lens import HookedTransformer, HookedTransformerConfig

# Build the TransformerLens config
tl_cfg = HookedTransformerConfig(
    n_layers           = N_LAYER,
    d_model            = N_EMBD,
    n_heads            = N_HEAD,
    d_head             = N_EMBD // N_HEAD,   # 128 // 4 = 32
    d_mlp              = 4 * N_EMBD,         # NanoGPT default: 4 * n_embd = 512
    n_ctx              = BLOCK_SIZE,
    d_vocab            = VOCAB_SIZE,
    act_fn             = "gelu_new",         # NanoGPT uses the 'new' GELU approximation
    normalization_type = "LN",
    tokenizer_name     = "gpt2",
    device             = "cpu",
    attn_only          = False,
)

def convert_nanogpt_weights(sd: dict, cfg: HookedTransformerConfig) -> dict:
    """
    Remap a NanoGPT state_dict to the TransformerLens HookedTransformer format.
    Handles the QKV split, Conv1D transpose, and per-head reshaping.
    """
    tl = {}
    d  = cfg.d_model
    nh = cfg.n_heads
    dh = cfg.d_head

    # Token + position embeddings
    tl["embed.W_E"]       = sd["transformer.wte.weight"]          # (vocab, d)
    tl["pos_embed.W_pos"] = sd["transformer.wpe.weight"]          # (ctx, d)

    # Final LayerNorm
    tl["ln_final.w"] = sd["transformer.ln_f.weight"]
    tl["ln_final.b"] = sd["transformer.ln_f.bias"]

    # Unembed head
    # NanoGPT may tie lm_head.weight == wte.weight
    lm_w = sd.get("lm_head.weight", sd["transformer.wte.weight"])
    tl["unembed.W_U"] = lm_w.T                                    # (d, vocab)
    tl["unembed.b_U"] = torch.zeros(cfg.d_vocab)

    for L in range(cfg.n_layers):
        p = f"transformer.h.{L}"

        # LayerNorm 1 (before attention)
        tl[f"blocks.{L}.ln1.w"] = sd[f"{p}.ln_1.weight"]
        tl[f"blocks.{L}.ln1.b"] = sd[f"{p}.ln_1.bias"]

        # ── Attention QKV ──────────────────────────────────────────────────
        # NanoGPT Conv1D stores (out_features, in_features)
        # c_attn.weight: (3*d, d)  ->  split into Q, K, V each (d, d)
        c_attn_w = sd[f"{p}.attn.c_attn.weight"]   # (3d, d)
        c_attn_b = sd[f"{p}.attn.c_attn.bias"]     # (3d,)

        W_Q_flat, W_K_flat, W_V_flat = c_attn_w.split(d, dim=0)   # each (d, d)
        b_Q_flat, b_K_flat, b_V_flat = c_attn_b.split(d, dim=0)   # each (d,)

        # Reshape to (n_heads, d_model, d_head)
        # W_Q_flat rows are the output directions (nh * dh), cols are input (d)
        # TL W_Q[h]: maps d_model -> d_head, stored as (nh, d, dh)
        tl[f"blocks.{L}.attn.W_Q"] = W_Q_flat.T.reshape(d, nh, dh).permute(1, 0, 2)
        tl[f"blocks.{L}.attn.W_K"] = W_K_flat.T.reshape(d, nh, dh).permute(1, 0, 2)
        tl[f"blocks.{L}.attn.W_V"] = W_V_flat.T.reshape(d, nh, dh).permute(1, 0, 2)

        tl[f"blocks.{L}.attn.b_Q"] = b_Q_flat.reshape(nh, dh)
        tl[f"blocks.{L}.attn.b_K"] = b_K_flat.reshape(nh, dh)
        tl[f"blocks.{L}.attn.b_V"] = b_V_flat.reshape(nh, dh)

        # ── Attention output projection ─────────────────────────────────────
        # c_proj.weight: (d, d) in Conv1D layout
        # TL W_O: (n_heads, d_head, d_model)
        c_proj_w = sd[f"{p}.attn.c_proj.weight"]   # (d, d)
        c_proj_b = sd[f"{p}.attn.c_proj.bias"]     # (d,)
        tl[f"blocks.{L}.attn.W_O"] = c_proj_w.T.reshape(nh, dh, d)
        tl[f"blocks.{L}.attn.b_O"] = c_proj_b

        # LayerNorm 2 (before MLP)
        tl[f"blocks.{L}.ln2.w"] = sd[f"{p}.ln_2.weight"]
        tl[f"blocks.{L}.ln2.b"] = sd[f"{p}.ln_2.bias"]

        # ── MLP ─────────────────────────────────────────────────────────────
        # NanoGPT Conv1D: c_fc.weight (4d, d), c_proj.weight (d, 4d)
        # TL convention: W_in (d, 4d), W_out (4d, d)
        tl[f"blocks.{L}.mlp.W_in"]  = sd[f"{p}.mlp.c_fc.weight"].T    # (d, 4d)
        tl[f"blocks.{L}.mlp.b_in"]  = sd[f"{p}.mlp.c_fc.bias"]        # (4d,)
        tl[f"blocks.{L}.mlp.W_out"] = sd[f"{p}.mlp.c_proj.weight"].T  # (4d, d)
        tl[f"blocks.{L}.mlp.b_out"] = sd[f"{p}.mlp.c_proj.bias"]      # (d,)

    return tl


print("Converting weights...")
tl_state_dict = convert_nanogpt_weights(raw_sd, tl_cfg)
print(f"Converted {len(tl_state_dict)} parameter tensors.")
print("Sample keys:", list(tl_state_dict.keys())[:8])

---
## Cell 5 — Build the HookedTransformer and Verify

In [ ]:
from transformer_lens import HookedTransformer

# Instantiate blank model, load converted weights
tl_model = HookedTransformer(tl_cfg)
tl_model.eval()

# strict=False tolerates missing mask buffers (causal mask etc.)
missing, unexpected = tl_model.load_state_dict(tl_state_dict, strict=False)

if missing:
    print("Missing keys (non-parameter buffers are expected and OK):")
    for k in missing[:10]:  print(f"  {k}")
if unexpected:
    print("Unexpected keys:")
    for k in unexpected[:10]: print(f"  {k}")

# Forward pass sanity check
test_tokens = torch.randint(0, VOCAB_SIZE, (1, 16))
with torch.no_grad():
    logits = tl_model(test_tokens)
assert logits.shape == (1, 16, VOCAB_SIZE), f"Unexpected logit shape: {logits.shape}"
print(f"Forward pass OK — logits shape: {logits.shape}")

# Verify MLP hook points exist (required by CLT-Forge's ActivationsStore)
print("\nMLP hook points:")
for L in range(N_LAYER):
    pre_key  = f"blocks.{L}.mlp.hook_pre"
    post_key = f"blocks.{L}.mlp.hook_post"
    has_pre  = pre_key  in tl_model.hook_dict
    has_post = post_key in tl_model.hook_dict
    print(f"  Layer {L}: hook_pre={has_pre}, hook_post={has_post}")
    assert has_pre and has_post, f"Missing MLP hook at layer {L}"

print("\nAll MLP hook points verified.")

# Save the converted weights for later re-use
torch.save(tl_state_dict, CONVERTED_MODEL_PATH)
print(f"Converted weights saved: {CONVERTED_MODEL_PATH}")

---
## Cell 6 — Move Model to Device and Freeze

In [ ]:
tl_model = tl_model.to(DEVICE)
tl_model.eval()

# Freeze the base model — we only extract activations, never train it
for param in tl_model.parameters():
    param.requires_grad_(False)

param_count = sum(p.numel() for p in tl_model.parameters())
print(f"Model on device   : {next(tl_model.parameters()).device}")
print(f"Total parameters  : {param_count:,}")

# Optional: quick generation check
try:
    with torch.no_grad():
        sample = tl_model.generate("The", max_new_tokens=8, temperature=1.0)
    print(f"Sample generation : {sample}")
except Exception as e:
    print(f"Generation check skipped: {e}")

---
## Cell 7 — Build the CLT-Forge Training Config

In [ ]:
from clt_forge import CLTTrainingRunnerConfig

gradient_accumulation_steps = GRAD_ACCUM_STEPS
effective_batch              = TRAIN_BATCH_SIZE_TOKENS * gradient_accumulation_steps
total_training_steps         = TOTAL_TRAINING_TOKENS // effective_batch

print(f"Effective batch size : {effective_batch} tokens")
print(f"Total training steps : {total_training_steps}")

cfg = CLTTrainingRunnerConfig(
    # System
    distributed_setup           = "none",          # 'feature_sharding' for multi-GPU
    device                      = DEVICE,
    dtype                       = "float32",       # use 'bfloat16' on Ampere+ GPUs with 1.5-2x lr
    seed                        = 42,

    # Model and data
    model_name                  = "gpt2",          # tells CLT-Forge which tokeniser to use
    d_in                        = N_EMBD,          # MLP input/output dimension = 128
    expansion_factor            = EXPANSION_FACTOR,
    dataset_path                = DATASET_PATH,
    context_size                = CONTEXT_SIZE,
    cached_activations_path     = CACHED_ACTIVATIONS_DIR,

    # Checkpointing
    n_checkpoints               = 5,
    checkpoint_path             = CLT_CHECKPOINT_DIR,
    from_pretrained_path        = None,            # set to resume from a checkpoint

    # JumpReLU sparsifying activation
    jumprelu_init_threshold     = 0.03,
    jumprelu_bandwidth          = 1.0,

    # Batching
    train_batch_size_tokens     = TRAIN_BATCH_SIZE_TOKENS,
    gradient_accumulation_steps = gradient_accumulation_steps,
    n_train_batch_per_buffer    = 64,

    # Training duration
    total_training_tokens       = TOTAL_TRAINING_TOKENS,

    # Optimisation
    lr                          = LEARNING_RATE,
    lr_warm_up_steps            = min(1_000, total_training_steps // 10),
    lr_decay_steps              = total_training_steps // 20,
    adam_beta1                  = 0.9,
    adam_beta2                  = 0.999,

    # Sparsity (L1 penalty)
    l0_coefficient              = L0_COEFFICIENT,
    l0_warm_up_steps            = int(0.7 * total_training_steps),
    dead_penalty_coef           = 1e-5,
    dead_feature_window         = 250,

    # Checkpoint selection
    checkpoint_l0               = [10, 5],
    optimal_l0                  = 5,

    # Logging
    log_to_wandb                = False,   # set True if wandb is configured
    wandb_project               = "nanogpt-clt",
    wandb_log_frequency         = 10,
    eval_every_n_wandb_logs     = 50,
    run_name                    = f"nanogpt-{N_LAYER}L-{N_EMBD}d-clt",
)

print("CLTTrainingRunnerConfig created successfully.")
print(f"  d_in             : {cfg.d_in}")
print(f"  expansion_factor : {cfg.expansion_factor}")
print(f"  total features   : {cfg.d_in * cfg.expansion_factor}")

---
## Cell 8 — Precompute and Cache MLP Activations

Runs the frozen NanoGPT over a large dataset and saves MLP input/output pairs to disk.
This step only needs to be done **once** per model.

The cell automatically skips if cached files already exist.

In [ ]:
import os
from clt_forge import ActivationsStore

existing = [f for f in os.listdir(CACHED_ACTIVATIONS_DIR)
            if f.endswith(".pt") or f.endswith(".zst")]

if existing:
    print(f"Found {len(existing)} cached activation files. Skipping generation.")
    print("To regenerate, delete the contents of:", CACHED_ACTIVATIONS_DIR)
else:
    print(f"Generating activations for {TOTAL_TRAINING_TOKENS:,} tokens...")
    print(f"  Context size  : {CONTEXT_SIZE}")
    print(f"  Compression   : {USE_COMPRESSION}")
    print(f"  Save path     : {CACHED_ACTIVATIONS_DIR}")

    store = ActivationsStore(
        model = tl_model,
        cfg   = cfg,
    )

    store.generate_and_save_activations(
        path            = cfg.cached_activations_path,
        use_compression = USE_COMPRESSION,
    )

    saved = [f for f in os.listdir(CACHED_ACTIVATIONS_DIR)
             if f.endswith(".pt") or f.endswith(".zst")]
    print(f"\nActivation caching complete. {len(saved)} files saved.")

---
## Cell 9 — Train the CLT

Trains the Cross-Layer Transcoder using the precomputed activation chunks.

The training objective minimises:
- **Reconstruction loss**: how well CLT outputs approximate MLP outputs across all layer pairs
- **Sparsity penalty (L1)**: encourages few active features per token (via JumpReLU)
- **Dead feature penalty**: prevents features from never activating

In [ ]:
import glob
from clt_forge import CLTTrainingRunner

print("Starting CLT training...")
print(f"  Layers x features : {N_LAYER} layers, {N_EMBD * EXPANSION_FACTOR} features")
print(f"  Total tokens      : {cfg.total_training_tokens:,}")
print(f"  Checkpoint dir    : {cfg.checkpoint_path}")
print()

trainer = CLTTrainingRunner(cfg)
trainer.run()

print("\nCLT training complete!")

# Find the latest checkpoint
checkpoints = sorted(glob.glob(os.path.join(CLT_CHECKPOINT_DIR, "**/*.pt"), recursive=True))
if checkpoints:
    BEST_CHECKPOINT = checkpoints[-1]
    print(f"Latest checkpoint  : {BEST_CHECKPOINT}")
else:
    print("WARNING: No checkpoint files found — check checkpoint_path in config.")
    BEST_CHECKPOINT = CLT_CHECKPOINT_DIR

---
## Cell 10 — Evaluate Reconstruction Quality

Computes per-layer explained variance: how much of the MLP output variance the CLT captures.

In [ ]:
import torch

try:
    from clt_forge import CLT

    clt_model = CLT.from_pretrained(BEST_CHECKPOINT)
    clt_model.eval().to(DEVICE)
    print(f"Loaded CLT from: {BEST_CHECKPOINT}")

    # Sample a small evaluation batch
    eval_tokens = torch.randint(0, VOCAB_SIZE, (4, CONTEXT_SIZE), device=DEVICE)

    with torch.no_grad():
        _, cache = tl_model.run_with_cache(
            eval_tokens,
            names_filter = lambda n: "mlp" in n and ("hook_pre" in n or "hook_post" in n)
        )

    mlp_inputs  = [cache[f"blocks.{L}.mlp.hook_pre"]  for L in range(N_LAYER)]
    mlp_outputs = [cache[f"blocks.{L}.mlp.hook_post"] for L in range(N_LAYER)]

    with torch.no_grad():
        clt_recons = clt_model.forward(mlp_inputs)  # list of [batch, seq, d_model] per layer

    print("\nPer-layer explained variance (higher is better, 1.0 = perfect):")
    total_ev = 0.0
    for L in range(N_LAYER):
        target = mlp_outputs[L].float()
        pred   = clt_recons[L].float()
        ss_res = ((target - pred) ** 2).sum().item()
        ss_tot = ((target - target.mean()) ** 2).sum().item()
        ev = 1.0 - ss_res / (ss_tot + 1e-8)
        total_ev += ev
        bar = '#' * int(max(0, ev) * 30)
        print(f"  Layer {L}: {ev:.4f}  |{bar:<30}|")
    print(f"\n  Mean EV: {total_ev / N_LAYER:.4f}")

except Exception as e:
    print(f"Evaluation skipped — checkpoint may not exist yet or API differs slightly.")
    print(f"Error: {e}")
    print("Re-run this cell after training completes.")

---
## Cell 11 — (Optional) Run AutoInterp

Generates natural-language descriptions for each CLT feature.

Set `RUN_AUTOINTERP = True` to enable.

In [ ]:
RUN_AUTOINTERP = False   # <-- set to True to run

if RUN_AUTOINTERP:
    from clt_forge import AutoInterp, AutoInterpConfig

    autointerp_cfg = AutoInterpConfig(
        device                  = DEVICE,
        model_name              = "gpt2",
        clt_path                = BEST_CHECKPOINT,
        latent_cache_path       = AUTOINTERP_DIR,
        dataset_path            = DATASET_PATH,
        context_size            = CONTEXT_SIZE,
        total_autointerp_tokens = 2_000_000,
        train_batch_size_tokens = 8192,
    )

    autointerp = AutoInterp(autointerp_cfg)
    autointerp.run(AUTOINTERP_DIR)
    print(f"AutoInterp results saved to: {AUTOINTERP_DIR}")
else:
    print("AutoInterp skipped. Set RUN_AUTOINTERP = True to enable.")

---
## Cell 12 — (Optional) Compute Attribution Graph

Set `RUN_ATTRIBUTION = True` and supply a prompt relevant to your training data.

In [ ]:
RUN_ATTRIBUTION    = False   # <-- set to True to run
ATTRIBUTION_PROMPT = 'The opposite of "large" is '  # change to a prompt from your domain

if RUN_ATTRIBUTION:
    from clt_forge import AttributionRunner

    runner = AttributionRunner(
        model_name = "gpt2",
        clt_path   = BEST_CHECKPOINT,
    )

    graph = runner.run(
        input_str = ATTRIBUTION_PROMPT,
        folder    = ATTRIBUTION_GRAPH_DIR,
    )

    print(f"Attribution graph saved to: {ATTRIBUTION_GRAPH_DIR}")
    print(f"Nodes: {len(graph.nodes)}, Edges: {len(graph.edges)}")
else:
    print("Attribution graph skipped. Set RUN_ATTRIBUTION = True to enable.")

---
## Cell 13 — (Optional) Launch Visual Interface

Starts a local Dash web server. Open **http://127.0.0.1:8050** in your browser after running.

Set `LAUNCH_UI = True` to enable.

In [ ]:
LAUNCH_UI = False   # <-- set to True to run (blocks until Ctrl+C)

if LAUNCH_UI:
    from clt_forge.frontend import main, AppConfig

    ui_cfg = AppConfig(
        graph_path      = ATTRIBUTION_GRAPH_DIR,
        autointerp_path = AUTOINTERP_DIR,
    )

    # Open http://127.0.0.1:8050 in your browser
    # Stop with Ctrl+C or interrupt the kernel
    main(ui_cfg)
else:
    print("Visual interface skipped. Set LAUNCH_UI = True to enable.")

---
## Summary

| Cell | Stage | Notes |
|------|-------|-------|
| 1 | Install dependencies | Run once |
| 2 | **Set paths and config** | Edit before first run |
| 3 | Load NanoGPT checkpoint | Handles DDP prefix, metadata |
| 4 | Convert weights to TL format | Full QKV split and reshape |
| 5 | Build and verify HookedTransformer | Forward pass + hook check |
| 6 | Move to device, freeze weights | Base model is never trained |
| 7 | Build CLT-Forge config | All hyperparameters here |
| 8 | Cache MLP activations | Skips if files already exist |
| 9 | **Train the CLT** | JumpReLU + L1 sparsity |
| 10 | Evaluate reconstruction | Explained variance per layer |
| 11 | AutoInterp | Optional; flag to enable |
| 12 | Attribution graph | Optional; flag to enable |
| 13 | Visual interface | Optional; flag to enable |

### Key parameters to tune

| Parameter | Cell | Guidance |
|-----------|------|----------|
| `EXPANSION_FACTOR` | 2 | Higher = more features, more memory. Start at 16 for n_embd=128 |
| `TOTAL_TRAINING_TOKENS` | 2 | More tokens = better features. Try 5M → 20M |
| `L0_COEFFICIENT` | 2 | Higher = sparser features. Range 1.0 – 4.0 |
| `LEARNING_RATE` | 2 | 4e-4 for fp32, 8e-4 for bfloat16 |
| `CONTEXT_SIZE` | 2 | Must be <= BLOCK_SIZE. Shorter = faster caching |